# 🔧 Module 3 — Class 5 Homework: Pipelines

**Topic:** Pipelines — ColumnTransformer, Pipeline, joblib

## 📋 What you have to do

This notebook **already has working code**. Your job:

1. **Run every cell from top to bottom** — it should run with no errors.
2. **Add a comment on EVERY code cell** that explains what the cell does, in your own words.
   - Use `#` to write comments inside the code cell.
   - Write at least 1 short sentence per cell. Longer is better.
3. **Save your notebook** as `Module<X>_Class<Y>_<YourName>.ipynb` and submit.

**Example of a good commented cell:**

```python
# Count how many customers churned vs stayed
# value_counts() returns the number of rows for each unique value
df['Churn'].value_counts()
```

**Example of a BAD comment (do not do this):**

```python
# count
df['Churn'].value_counts()
```

---


In [ ]:
# === SETUP — run this first ===
# Import essential libraries for data analysis and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Silence warnings to keep the notebook clean
import warnings; warnings.filterwarnings('ignore')

# URL of the dataset (IBM Telco Customer Churn)
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'

# Load the dataset from the URL into a DataFrame
df = pd.read_csv(url)

# Convert 'TotalCharges' column to numeric, fixing blank spaces with 0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Print the final shape of the loaded data
print('Loaded:', df.shape)

Loaded: (7043, 21)


### Cell 1 — set up the target and features

In [ ]:
# Map text labels ('No'/'Yes') to binary integers (0/1) for the target variable
df['Churn_bin'] = df['Churn'].map({'No': 0, 'Yes': 1})

# List out the continuous numerical columns to include
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

# List out the categorical columns to include
cat_features = ['gender', 'Contract', 'PaymentMethod']

# Combine both feature lists to create the full input DataFrame (X)
X = df[num_features + cat_features]

# Isolate the target label column (y)
y = df['Churn_bin']

# Print the shapes of X and y to verify the alignment and row counts
print('X:', X.shape, 'y:', y.shape)

X: (7043, 6) y: (7043,)


### Cell 2 — split into train and test

In [ ]:
# Import the function to split data into random train and test subsets
from sklearn.model_selection import train_test_split

# Split X and y into 80% training data and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Print the final sizes of both training and testing datasets
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (5634, 6), Test: (1409, 6)


### Cell 3 — build the numeric preprocessor

In [ ]:
# Import Pipeline and preprocessing tools from sklearn
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Define a pipeline specifically for cleaning and scaling numerical features
num_pipe = Pipeline([
    # Step 1: Fill any missing values using the median of that column
    ('imputer', SimpleImputer(strategy='median')),

    # Step 2: Scale the features to have a mean of 0 and variance of 1
    ('scaler', StandardScaler())
])

# Display the visual diagram of the pipeline structure
num_pipe

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

### Cell 4 — build the categorical preprocessor

In [ ]:
# Import OneHotEncoder to transform categorical text into binary numbers
from sklearn.preprocessing import OneHotEncoder

# Define a pipeline specifically for cleaning and encoding categorical features
cat_pipe = Pipeline([
    # Step 1: Fill missing text values with the most common value (mode) of that column
    ('imputer', SimpleImputer(strategy='most_frequent')),

    # Step 2: Convert text categories into 0 and 1 columns, ignoring any new unseen labels
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Display the visual diagram of the categorical pipeline structure
cat_pipe

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

### Cell 5 — combine both with ColumnTransformer

In [ ]:
# Import ColumnTransformer to apply different pipelines to specific columns
from sklearn.compose import ColumnTransformer

# Bundle the numerical and categorical pipelines together into a single preprocessor
preprocessor = ColumnTransformer([
    # Apply the numerical pipeline to the designated continuous columns
    ('num', num_pipe, num_features),

    # Apply the categorical pipeline to the designated text columns
    ('cat', cat_pipe, cat_features)
])

# Display the final, combined preprocessing architecture diagram
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['gender', 'Contract', 'PaymentMethod'])])

### Cell 6 — full pipeline: preprocessor + model

In [ ]:
# Import Logistic Regression classifier from scikit-learn
from sklearn.linear_model import LogisticRegression

# Create the final end-to-end pipeline combining preprocessing and modeling
full_pipe = Pipeline([
    # Step 1: Clean, scale, and encode all input data using our preprocessor
    ('preprocess', preprocessor),

    # Step 2: Feed the cleaned data directly into the Logistic Regression model
    ('model', LogisticRegression(max_iter=1000))
])

# Display the interactive visual diagram of the entire machine learning pipeline
full_pipe

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['gender', 'Contract',
                                                   'PaymentMethod'])])),
                ('model', LogisticRegression(max_iter=1000))])

### Cell 7 — fit on training data

In [ ]:
# Train the entire pipeline (preprocessing + logistic regression) on the training data
full_pipe.fit(X_train, y_train)

# Print a success message once the training process is complete
print('✅ Pipeline trained')

✅ Pipeline trained


### Cell 8 — evaluate on test data

In [ ]:
# Import the metric to calculate classification accuracy
from sklearn.metrics import accuracy_score

# Use the trained pipeline to make predictions on the unseen test dataset
y_pred = full_pipe.predict(X_test)

# Compare the model's predictions (y_pred) with the actual true labels (y_test)
acc = accuracy_score(y_test, y_pred)

# Print the final accuracy score formatted to 4 decimal places
print(f'Test accuracy: {acc:.4f}')

Test accuracy: 0.7963


### Cell 9 — save the trained pipeline

In [ ]:
# Import joblib for efficient serialization of large numpy arrays and models
import joblib

# Save the entire trained pipeline to a local file named 'churn_pipeline.joblib'
joblib.dump(full_pipe, 'churn_pipeline.joblib')

# Load the saved pipeline file back into a new object called 'loaded'
loaded = joblib.load('churn_pipeline.joblib')

# Print a confirmation message indicating the save/load process was successful
print('Saved and loaded.')

# Directly evaluate and print the test accuracy of the reloaded pipeline
print('Test accuracy from loaded model:', round(loaded.score(X_test, y_test), 4))

Saved and loaded.
Test accuracy from loaded model: 0.7963
